## Deutsch–Jozsa Algorithm (n-qubit)

Deutsch–Jozsa generalizes Deutsch's algorithm from a single input bit to n input bits. Now f: {0,1}ⁿ → {0,1}, and you're promised it's either **constant** (same output for all 2ⁿ inputs) or **balanced** (outputs 0 for exactly half the inputs and 1 for the other half). The task is still to figure out which, using the oracle as few times as possible.

Classically, in the worst case you might need to check just over half the inputs, $2^{n-1}+1$, before you can be sure whether f is constant or balanced — cost that grows exponentially with n. Deutsch–Jozsa still solves it with exactly **one** oracle query, no matter how large n gets, so the quantum advantage grows dramatically as n increases.

**Circuit:** the setup is the same as the single-bit case, just applied to n input qubits at once. All n input qubits are Hadamard'd into an equal superposition over all $2^n$ possible input strings, and the ancilla is prepared in the $(|0\rangle-|1\rangle)/\sqrt{2}$ state. The oracle applies phase kickback to every input string in the superposition simultaneously: each basis state $|x\rangle$ picks up a factor of $(-1)^{f(x)}$. A final layer of Hadamards on all n input qubits is applied before measurement.

**Why the all-zeros outcome signals constant.** After the final Hadamards, the amplitude of measuring all n qubits as 0 works out to $\frac{1}{2^n}\sum_x (-1)^{f(x)}$, which is just the average of $(-1)^{f(x)}$ across all inputs. If f is constant, every term in that sum has the same sign, so they all add up and the amplitude becomes exactly ±1: you measure all-zeros with certainty. If f is balanced, exactly half the terms are +1 and half are −1, so they cancel completely and the amplitude on all-zeros is 0: you're guaranteed to see at least one qubit as 1.

**Decision rule.** Measuring all zeros means f is constant. Measuring anything else means f is balanced.

**Oracles used here.** The constant oracles apply either no gates (f(x)=0 for all x) or an unconditional X on the ancilla (f(x)=1 for all x). The balanced oracle applies a CNOT from every input qubit to the ancilla, so the ancilla ends up flipped based on the parity of the input string (the XOR of all its bits). Parity is balanced for any n: flipping any single fixed bit of an input turns every even-parity string into an odd-parity one and vice versa, pairing them up exactly, so half of all $2^n$ inputs have even parity and half have odd.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

def constant_oracle(n):
    oracle = QuantumCircuit(n + 1)
    # f(x) = 0 for all x: no gates needed, ancilla never flips
    return oracle

def constant_oracle_one(n):
    oracle = QuantumCircuit(n + 1)
    oracle.x(n)  # f(x) = 1 for all x: ancilla always flips, unconditionally
    return oracle

def balanced_oracle(n):
    oracle = QuantumCircuit(n + 1)
    for i in range(n):
        oracle.cx(i, n)  # Flip ancilla based on parity (XOR) of all input qubits
    return oracle

def deutsch_jozsa(oracle_circuit, n):
    qc = QuantumCircuit(n + 1, n)  # n input qubits + 1 ancilla, n classical bits (only measure inputs)
    qc.x(n)               # Prepare ancilla in |1⟩
    qc.h(n)                # Ancilla: (|0⟩-|1⟩)/√2
    qc.h(range(n))          # All input qubits: equal superposition over all 2^n input strings
    qc.compose(oracle_circuit, inplace=True)  # Apply oracle: phase kickback on each input basis state
    qc.h(range(n))          # Final Hadamard layer on input qubits before measurement
    qc.measure(range(n), range(n))
    return qc

# Verification n = 2
n = 2
simulator = AerSimulator()

qc_const = deutsch_jozsa(constant_oracle(n), n)
result_const = simulator.run(qc_const).result()
print("n=2, Constant f(x)=0:", result_const.get_counts())

qc_const_one = deutsch_jozsa(constant_oracle_one(n), n)
result_const_one = simulator.run(qc_const_one).result()
print("n=2, Constant f(x)=1:", result_const_one.get_counts())

qc_bal = deutsch_jozsa(balanced_oracle(n), n)
result_bal = simulator.run(qc_bal).result()
print("n=2, Balanced:", result_bal.get_counts())


# Verification n = 3
n = 3
simulator = AerSimulator()

qc_const = deutsch_jozsa(constant_oracle(n), n)
result_const = simulator.run(qc_const).result()
print("n=3, Constant f(x)=0:", result_const.get_counts())

qc_const_one = deutsch_jozsa(constant_oracle_one(n), n)
result_const_one = simulator.run(qc_const_one).result()
print("n=3, Constant f(x)=1:", result_const_one.get_counts())

qc_bal = deutsch_jozsa(balanced_oracle(n), n)
result_bal = simulator.run(qc_bal).result()
print("n=3, Balanced:", result_bal.get_counts())


# Verification n = 4
n = 4
simulator = AerSimulator()

qc_const = deutsch_jozsa(constant_oracle(n), n)
result_const = simulator.run(qc_const).result()
print("n=4, Constant f(x)=0:", result_const.get_counts())

qc_const_one = deutsch_jozsa(constant_oracle_one(n), n)
result_const_one = simulator.run(qc_const_one).result()
print("n=4, Constant f(x)=1:", result_const_one.get_counts())

qc_bal = deutsch_jozsa(balanced_oracle(n), n)
result_bal = simulator.run(qc_bal).result()
print("n=4, Balanced:", result_bal.get_counts())

n=2, Constant f(x)=0: {'00': 1024}
n=2, Constant f(x)=1: {'00': 1024}
n=2, Balanced: {'11': 1024}
n=3, Constant f(x)=0: {'000': 1024}
n=3, Constant f(x)=1: {'000': 1024}
n=3, Balanced: {'111': 1024}
n=4, Constant f(x)=0: {'0000': 1024}
n=4, Constant f(x)=1: {'0000': 1024}
n=4, Balanced: {'1111': 1024}


**Results (1024 shots each, n=2,3,4):**

Every constant oracle measured all-zeros with certainty, at every value of n tested: `00`, `000`, and `0000` respectively, each with 1024/1024 shots. Every balanced oracle measured all-ones with certainty: `11`, `111`, and `1111`, again 1024/1024 shots.

The all-ones outcome specifically (rather than just "not all-zeros") is a consequence of using the parity oracle: since f here is the XOR of all input bits, the resulting measurement pattern happens to be all-ones for this particular balanced function. A different balanced oracle would still guarantee measuring at least one 1, but not necessarily this exact all-ones pattern.

These results confirm the algorithm correctly distinguishes constant from balanced functions using a single oracle query, regardless of n.